In [1]:
# from src.find_bids.models.classify.features import get_features_from_series, get_features_from_db
from src.find_bids.models.classify.core import ml_prep_pipeline
from src.find_bids.models.extract.dataset import Dataset
from src.find_bids.models.extract.series import initialize_features_db, SeriesFeatures, get_features_from_db
from src.find_bids.models.infer.core import DatasetsInference

from upath import UPath

import pandas as pd


In [2]:
# features_root: UPath = UPath("ssh://seadragon/rsrch5/home/csi/Quarles_Lab/find_BIDS/features")
# df_features: pd.DataFrame = get_features_from_series(
#     features_root=features_root,
#     load_existing=False,
#     save=True,
#     sample_subjects=1,
#     rebuild_old_root="/rsrch5/home/csi",
#     rebuild_new_root="ssh://seadragon/rsrch5/home/csi")
# df_features.head()

In [ ]:
DB_PATH = UPath("/Volumes/csi/Quarles_Lab/find_BIDS/features/features.db")
root_dir = UPath("/Volumes/csi/Quarles_Lab/")
proactive_dataset_features_path = root_dir / "find_BIDS/features/PROACTIVE/dataset.json"
qiac_dataset_features_path = root_dir / "find_BIDS/features/QIAC/dataset.json"

if proactive_dataset_features_path.exists():
    proactive_dataset = Dataset.from_json(proactive_dataset_features_path)
else:
    raise FileNotFoundError(f"Proactive dataset features not found at {proactive_dataset_features_path}")

if qiac_dataset_features_path.exists():
    qiac_dataset = Dataset.from_json(qiac_dataset_features_path)
else:
    raise FileNotFoundError(f"QIAC dataset features not found at {qiac_dataset_features_path}")

In [4]:
import importlib
from src.find_bids.models.extract import series as extract_series

importlib.reload(extract_series)
get_features_from_db = extract_series.get_features_from_db

features_root: UPath = UPath("/Volumes/csi/Quarles_Lab/find_BIDS/features")
features_from_db = get_features_from_db(
    db_path=DB_PATH,
    save=True,
    load_existing=True,
    to_series_features=False,
    datasets=[("proactive", proactive_dataset), ("qiac", qiac_dataset)],
 )

if isinstance(features_from_db, pd.DataFrame):
    raw_df = features_from_db.copy()
else:
    raw_df = pd.DataFrame(features_from_db)

# get_features_from_db can return rows with JSON payload in `data` and key columns in the index.
if "data" in raw_df.columns:
    key_df = raw_df.reset_index()
    rebuilt_rows = []
    for _, row in key_df.iterrows():
        sf = SeriesFeatures.from_json_str(row["data"])
        flat = sf.flatten()
        flat["subject_id"] = row.get("subject_id")
        flat["session_id"] = row.get("session_id")
        flat["series_uid"] = row.get("series_uid")
        flat["dataset"] = row.get("dataset")
        rebuilt_rows.append(flat)
    df_features_from_db = pd.DataFrame(rebuilt_rows)
else:
    df_features_from_db = raw_df.copy()

if "series_id" not in df_features_from_db.columns and "series_uid" in df_features_from_db.columns:
    df_features_from_db = df_features_from_db.rename(columns={"series_uid": "series_id"})

df_features_from_db["dataset"] = df_features_from_db["dataset"].replace({"xnat": "proactive", "Raw_MRI": "qiac"})
print("Feature DF shape:", df_features_from_db.shape)
print("Has required merge keys:", all(c in df_features_from_db.columns for c in ["dataset", "subject_id", "session_id", "series_id"]))
print("Missing dataset rows:", int(df_features_from_db["dataset"].isna().sum()))
df_features_from_db[["dataset", "subject_id", "session_id", "series_id"]].head()

Loading existing features from /Volumes/csi/Quarles_Lab/find_BIDS/features/all_series_features_from_db.csv
Feature DF shape: (77489, 86)
Has required merge keys: True
Missing dataset rows: 86


,dataset,subject_id,session_id,series_id
0,qiac,1012-3929,1012-3929-3589-0650,1.3.12.2.1107.5.2.18.141126.300000250124100121...
1,qiac,1012-3929,1012-3929-3589-0650,1.3.12.2.1107.5.2.18.141126.300000250124100121...
2,qiac,1012-3929,1012-3929-3589-0650,1.3.12.2.1107.5.2.18.141126.300000250124100810...
3,qiac,1012-3929,1012-3929-3589-0650,1.3.12.2.1107.5.2.18.141126.300000250124100810...
4,qiac,1012-3929,1012-3929-3589-0650,1.3.12.2.1107.5.2.18.141126.300000250124100810...


In [5]:
RELOAD: bool = True  # Set to True to regenerate the full dataset labels, False to load from CSV if it exists
if UPath("all_heuristic_scores.csv").exists() and not RELOAD:
    data_inference_all = DatasetsInference.from_csv("all_heuristic_scores.csv")
    df_inference_all = data_inference_all.to_dataframe()
else:
    # data_inference_all = DatasetsInference.from_datasets([proactive_dataset, qiac_dataset], sample_subjects_per_subjects=None)
    data_inference_all = DatasetsInference.from_db(datasets=[proactive_dataset, qiac_dataset], db_path=DB_PATH)
    data_inference_all.to_csv("all_heuristic_scores.csv")
    df_inference_all = data_inference_all.to_dataframe()
df_inference_all['dataset'] = df_inference_all['dataset'].replace({'xnat': 'proactive', 'Raw_MRI': 'qiac'})
df_inference_all.head()

Loading existing features from /Volumes/csi/Quarles_Lab/find_BIDS/features/all_series_features_from_db.csv


,dataset,subject_id,session_id,series_id,series_description,inferred_datatype,datatype_confidence,inferred_suffix,suffix_confidence,is_derived,derived_confidence,label,min_confidence
0,proactive,1022-1937,1022-1937-2284-6740,1.3.12.2.1107.5.2.18.52107.2022112911590867089...,3Pl Loc_11,localizer,0.999649,unknown,0.000000,False,0.582783,localizer_unknown,0.000000
1,proactive,1022-1937,1022-1937-2284-6740,1.3.12.2.1107.5.2.18.52107.2022112912032862322...,Ax T2,anat,0.953943,T2w,0.999149,False,0.823201,anat_T2w,0.823201
2,proactive,1022-1937,1022-1937-2284-6740,1.3.12.2.1107.5.2.18.52107.2022112912070827969...,Ax FLAIR,anat,0.525680,FLAIR,0.994938,False,0.823201,anat_FLAIR,0.525680
3,proactive,1022-1937,1022-1937-2284-6740,1.3.12.2.1107.5.2.18.52107.2022112912121751467...,Ax T2 Propeller,anat,0.525680,T2w,0.999149,False,0.823201,anat_T2w,0.525680
4,proactive,1022-1937,1022-1937-2284-6740,1.3.12.2.1107.5.2.18.52107.2022112912150182192...,Ax T2star,anat,0.717083,T2starw,0.976321,False,0.823201,anat_T2starw,0.717083


In [6]:
# Safety fallback: if dataset was not recoverable from extracted features, map from heuristic keys.
if "dataset" not in df_features_from_db.columns or df_features_from_db["dataset"].isna().all():
    key_map = df_inference_all[["dataset", "subject_id", "session_id", "series_id"]].drop_duplicates()
    df_features_for_ml = df_features_from_db.drop(columns=["dataset"], errors="ignore").merge(
        key_map,
        on=["subject_id", "session_id", "series_id"],
        how="inner",
    )
else:
    df_features_for_ml = df_features_from_db.copy()

sets = ml_prep_pipeline(
    series_features_df=df_features_for_ml,
    heuristic_scores_df=df_inference_all
)
sets["test"][0]

/Users/cmokashi/Documents/GitHub/find_BIDS/src/find_bids/models/classify/engineer.py:553: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[list(keyword_df.columns)] = keyword_df


,modality,series_number,instances,num_instances,num_unique_slices,num_volumes,field_strength,rows,columns,num_slices,...,tfidf_sequence_name_l3d,tfidf_sequence_name_l3d1,tfidf_sequence_name_l3d1_1,tfidf_sequence_name_ns_2,tfidf_sequence_name_pfi,tfidf_sequence_name_pfid,tfidf_sequence_name_pfid2,tfidf_sequence_name_se2_1,tfidf_sequence_name_se2d_1,tfidf_sequence_name_se2d1
1,mr,11.0,[1.3.12.2.1107.5.2.18.141126.30000025012410012...,35,35,1,1.5,128.0,128.0,35.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
18,mr,22.0,[1.3.12.2.1107.5.2.19.145569.30000025042312270...,30,30,1,3.0,768.0,624.0,30.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.295503,0.295503,0.295625
26,mr,6.0,[1.3.12.2.1107.5.2.19.145569.30000025042312124...,30,30,1,3.0,256.0,256.0,30.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
38,mr,13.0,[1.3.12.2.1107.5.2.18.142587.30000025073007560...,30,30,1,1.5,512.0,464.0,30.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
41,mr,17.0,[1.3.12.2.1107.5.2.18.142587.30000025073007560...,26,26,1,1.5,128.0,128.0,26.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77362,mr,7.0,[1.3.12.2.1107.5.2.43.166019.20221208091643216...,31,1,1,3.0,896.0,896.0,1.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
77381,mr,100.0,[1.3.12.2.1107.5.2.19.145569.30000023040316171...,9,9,1,3.0,64.0,64.0,9.0,...,0.0,0.0,0.0,0.0,0.23128,0.231404,0.231404,0.000000,0.000000,0.000000
77384,mr,12.0,[1.3.12.2.1107.5.2.19.145569.20230403115154147...,30,30,1,3.0,640.0,580.0,30.0,...,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
77386,mr,14.0,[1.3.12.2.1107.5.2.19.145569.20230403120436242...,176,176,1,3.0,512.0,512.0,176.0,...,0.0,0.0,0.0,1.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
train_X, train_y = sets["train"]
test_X, test_y = sets["test"]
val_X, val_y = sets["val"]

In [8]:
train_X.columns.tolist()

['modality',
 'series_number',
 'instances',
 'num_instances',
 'num_unique_slices',
 'num_volumes',
 'field_strength',
 'rows',
 'columns',
 'num_slices',
 'geometry_hash',
 'repetition_time',
 'echo_time',
 'inversion_time',
 'flip_angle',
 'num_timepoints',
 'temporal_variation',
 'is_3D',
 'is_isotropic',
 'num_b0',
 'num_diffusion_volumes',
 'num_diffusion_directions',
 'has_diffusion',
 'temporal_spacing',
 'slice_thickness',
 'spacing_between_slices',
 'echo_spacing',
 'multiband_factor',
 'is_original',
 'is_primary',
 'is_mpr',
 'is_mip',
 'is_projection',
 'is_reformatted',
 'has_diffusion_token',
 'has_perfusion_token',
 'is_adc',
 'is_fa',
 'is_trace',
 'is_cbf',
 'is_cbv',
 'is_magnitude',
 'is_phase',
 'voxel_x',
 'voxel_y',
 'voxel_z',
 'pixel_spacing_x',
 'pixel_spacing_y',
 'bvals_count',
 'bvals_min',
 'bvals_max',
 'bvals_mean',
 'echo_times_count',
 'echo_times_min',
 'echo_times_max',
 'echo_times_mean',
 'echo_numbers_count',
 'echo_numbers_min',
 'echo_numbers_ma

In [9]:
len(train_X.columns.tolist())

552

## Modeling: Dataset Configurations, Training, and Evaluation

This section builds two dataset regimes and compares models:

1. no confidence-tier stratification and no confidence weights
2. confidence-tier stratification with continuous confidence-based sample weights

Models evaluated: Random Forest and XGBoost (with a sklearn fallback if XGBoost is unavailable).

In [12]:
import importlib

from src.find_bids.models.classify import core as classify_core

importlib.reload(classify_core)
ml_prep_pipeline_with_confidence_options = classify_core.ml_prep_pipeline_with_confidence_options

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False

print(f"XGBoost available: {HAS_XGBOOST}")

XGBoost available: False


In [13]:
from time import perf_counter

required_vars = ["df_features_for_ml", "df_inference_all"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(
        "Missing prerequisite variables for dataset preparation: "
        + ", ".join(missing_vars)
        + ". Re-run the data loading cells before this cell."
    )

prep_start = perf_counter()
print("[dataset-prep] Building no-tier/no-weight split...")
sets_no_tier = ml_prep_pipeline_with_confidence_options(
    series_features_df=df_features_for_ml,
    heuristic_scores_df=df_inference_all,
    use_confidence_tier_stratification=False,
    confidence_weighting_mode="none",
)

print("[dataset-prep] Building tier-stratified/weighted split...")
sets_tier_weighted = ml_prep_pipeline_with_confidence_options(
    series_features_df=df_features_for_ml,
    heuristic_scores_df=df_inference_all,
    use_confidence_tier_stratification=True,
    confidence_weighting_mode="continuous",
    confidence_weight_min=0.3,
    confidence_weight_gamma=1.5,
)

def summarize_set_shapes(sets_with_weights: dict[str, tuple[pd.DataFrame, pd.DataFrame, pd.Series]], label: str) -> pd.DataFrame:
    rows = []
    for split_name in ["train", "val", "test"]:
        X, y, w = sets_with_weights[split_name]
        rows.append(
            {
                "regime": label,
                "split": split_name,
                "rows": len(X),
                "n_features": X.shape[1],
                "mean_weight": float(np.mean(w)),
                "min_weight": float(np.min(w)),
                "max_weight": float(np.max(w)),
                "datatype_classes": int(y["inferred_datatype"].nunique()),
                "suffix_classes": int(y["inferred_suffix"].nunique()),
            }
        )
    return pd.DataFrame(rows)

shape_summary = pd.concat(
    [
        summarize_set_shapes(sets_no_tier, "no_tier_no_weights"),
        summarize_set_shapes(sets_tier_weighted, "tier_stratified_weighted"),
    ],
    ignore_index=True,
)

print(f"[dataset-prep] Completed in {perf_counter() - prep_start:.2f}s")
shape_summary

[dataset-prep] Building no-tier/no-weight split...


/Users/cmokashi/Documents/GitHub/find_BIDS/src/find_bids/models/classify/engineer.py:553: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[list(keyword_df.columns)] = keyword_df


[dataset-prep] Building tier-stratified/weighted split...


/Users/cmokashi/Documents/GitHub/find_BIDS/src/find_bids/models/classify/engineer.py:553: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[list(keyword_df.columns)] = keyword_df


[dataset-prep] Completed in 18.14s


,regime,split,rows,n_features,mean_weight,min_weight,max_weight,datatype_classes,suffix_classes
0,no_tier_no_weights,train,54182,553,1.000000,1.0,1.0,7,28
1,no_tier_no_weights,val,11610,553,1.000000,1.0,1.0,7,25
2,no_tier_no_weights,test,11611,553,1.000000,1.0,1.0,7,25
3,tier_stratified_weighted,train,54181,552,0.641333,0.3,1.0,7,28
4,tier_stratified_weighted,val,11610,552,0.642594,0.3,1.0,7,25
5,tier_stratified_weighted,test,11612,552,0.641624,0.3,1.0,7,22


In [ ]:
from time import perf_counter

def _build_models(random_state: int = 42, speed_profile: str = "quick") -> dict[str, object]:
    speed_profile = speed_profile.lower().strip()
    rf_estimators = 200
    xgb_estimators = 250
    xgb_max_depth = 6
    if speed_profile == "balanced":
        rf_estimators = 350
        xgb_estimators = 400
        xgb_max_depth = 8
    elif speed_profile == "thorough":
        rf_estimators = 500
        xgb_estimators = 700
        xgb_max_depth = 10

    models: dict[str, object] = {
        "RandomForest": RandomForestClassifier(
            n_estimators=rf_estimators,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=random_state,
        )
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=xgb_estimators,
            max_depth=xgb_max_depth,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_jobs=-1,
            random_state=random_state,
            tree_method="hist",
        )
    return models


def _normalize_unhashable_category_values(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    normalized = frame.copy()

    def to_hashable_text(value: object) -> object:
        if isinstance(value, (list, tuple, set)):
            return "|".join(map(str, value))
        return value

    for col in cols:
        normalized[col] = normalized[col].map(to_hashable_text)
    return normalized


def _prepare_model_matrices(train_X: pd.DataFrame, val_X: pd.DataFrame, test_X: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cat_cols = train_X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    if not cat_cols:
        return train_X, val_X, test_X

    train_norm = _normalize_unhashable_category_values(train_X, cat_cols)
    val_norm = _normalize_unhashable_category_values(val_X, cat_cols)
    test_norm = _normalize_unhashable_category_values(test_X, cat_cols)

    train_enc = pd.get_dummies(train_norm, columns=cat_cols, dummy_na=True, dtype=np.int8)
    val_enc = pd.get_dummies(val_norm, columns=cat_cols, dummy_na=True, dtype=np.int8)
    test_enc = pd.get_dummies(test_norm, columns=cat_cols, dummy_na=True, dtype=np.int8)

    val_enc = val_enc.reindex(columns=train_enc.columns, fill_value=0)
    test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)
    return train_enc, val_enc, test_enc


def _evaluate_regime(
    sets_with_weights: dict[str, tuple[pd.DataFrame, pd.DataFrame, pd.Series]],
    regime_name: str,
    random_state: int = 42,
    speed_profile: str = "quick",
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[tuple[str, str], object]]:
    train_X, train_y, train_w = sets_with_weights["train"]
    val_X, val_y, _ = sets_with_weights["val"]
    test_X, test_y, _ = sets_with_weights["test"]

    train_X, val_X, test_X = _prepare_model_matrices(train_X, val_X, test_X)
    models = _build_models(random_state=random_state, speed_profile=speed_profile)
    targets = ["inferred_datatype", "inferred_suffix"]
    total_jobs = len(models) * len(targets)

    if verbose:
        print(f"[{regime_name}] Starting evaluation")
        print(f"[{regime_name}] Speed profile: {speed_profile}")
        print(f"[{regime_name}] Train/Val/Test rows: {len(train_X)}/{len(val_X)}/{len(test_X)}")
        print(f"[{regime_name}] Features after encoding: {train_X.shape[1]}")
        print(f"[{regime_name}] Jobs to run: {total_jobs} ({len(targets)} targets x {len(models)} models)")

    rows: list[dict] = []
    trained_models: dict[tuple[str, str], object] = {}

    regime_start = perf_counter()
    job_idx = 0
    for target in targets:
        y_train = train_y[target].astype("string")
        y_val = val_y[target].astype("string")
        y_test = test_y[target].astype("string")

        for model_name, model in models.items():
            job_idx += 1
            fit_kwargs = {}
            if regime_name == "tier_stratified_weighted":
                fit_kwargs["sample_weight"] = train_w.to_numpy()

            if verbose:
                print(f"[{regime_name}] [{job_idx}/{total_jobs}] Fitting {model_name} for {target}...")
            fit_start = perf_counter()
            model.fit(train_X, y_train, **fit_kwargs)
            fit_seconds = perf_counter() - fit_start
            trained_models[(target, model_name)] = model
            if verbose:
                print(f"[{regime_name}] [{job_idx}/{total_jobs}] Done in {fit_seconds:.2f}s")

            for split_name, X_split, y_true in [
                ("val", val_X, y_val),
                ("test", test_X, y_test),
            ]:
                pred_start = perf_counter()
                y_pred = pd.Series(model.predict(X_split), index=y_true.index).astype("string")
                pred_seconds = perf_counter() - pred_start
                rows.append(
                    {
                        "regime": regime_name,
                        "target": target,
                        "model": model_name,
                        "split": split_name,
                        "n_rows": len(y_true),
                        "fit_seconds": float(fit_seconds),
                        "predict_seconds": float(pred_seconds),
                        "accuracy": float(accuracy_score(y_true, y_pred)),
                        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
                        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
                    }
                )

    total_seconds = perf_counter() - regime_start
    if verbose:
        print(f"[{regime_name}] Completed in {total_seconds:.2f}s")

    return pd.DataFrame(rows), trained_models

In [ ]:
required_vars = ["sets_no_tier", "sets_tier_weighted"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(
        "Missing prepared dataset sets: "
        + ", ".join(missing_vars)
        + ". Re-run the dataset preparation cell first."
    )

overall_start = perf_counter()
print("[training] Starting model evaluation across regimes...")

results_no_tier, models_no_tier = _evaluate_regime(
    sets_no_tier,
    regime_name="no_tier_no_weights",
    speed_profile="quick",
    verbose=True,
    random_state=42,
 )

results_tier_weighted, models_tier_weighted = _evaluate_regime(
    sets_tier_weighted,
    regime_name="tier_stratified_weighted",
    speed_profile="quick",
    verbose=True,
    random_state=42,
 )

results_all = pd.concat([results_no_tier, results_tier_weighted], ignore_index=True)
results_all = results_all.sort_values(["target", "split", "macro_f1", "weighted_f1"], ascending=[True, True, False, False])

runtime_summary = (
    results_all.groupby(["regime", "target", "model"], as_index=False)[["fit_seconds", "predict_seconds"]]
    .mean()
    .sort_values(["regime", "fit_seconds"], ascending=[True, False])
)

print(f"[training] Total experiment wall time: {perf_counter() - overall_start:.2f}s")
print("[training] Mean fit/predict time per model:")
runtime_summary

results_all

NameError: name 'sets_no_tier' is not defined

In [ ]:
summary = (
    results_all
    .groupby(["target", "model", "regime", "split"], as_index=False)[["accuracy", "macro_f1", "weighted_f1"]]
    .mean()
)

pivot_macro = summary.pivot_table(
    index=["target", "model", "split"],
    columns="regime",
    values="macro_f1",
)

for required in ["no_tier_no_weights", "tier_stratified_weighted"]:
    if required not in pivot_macro.columns:
        pivot_macro[required] = np.nan

pivot_macro["delta_macro_f1_weighted_minus_plain"] = (
    pivot_macro["tier_stratified_weighted"] - pivot_macro["no_tier_no_weights"]
)

pivot_macro.sort_values("delta_macro_f1_weighted_minus_plain", ascending=False)